<!-- source: new + PRZ[1] -->
# M0 · Start: dane TechRetail w Twoim workspace

**Przebieg:** wstęp i historia, krótkie demo, lab i Canvas agenta

Jesteś analitykiem w **TechRetail Corp**, dystrybutorze elektroniki dla firm B2B. Zarząd chce asystenta, który odpowiada na pytania o klientów. Część odpowiedzi jest w tabeli, a część w raportach PDF analityków. Przez cały dzień budujemy tego asystenta: od rozmowy w AI Playground do agenta, który sam wybiera między SQL a dokumentami.

Ten notebook przygotowuje wszystko, czego potrzebują moduły M1–M6:

| Krok | Co powstaje |
|---|---|
| 1 | tabela `workspace.default.gold_customer_360`: 28 813 klientów, 19 kolumn |
| 2 | Volume `retail_docs` z 10 raportami PDF |
| 3 | tabela `retail_rag_chunks` z gotowymi fragmentami raportów |
| 4 | start endpointu AI Search `retail_rag_search` (w tle, gotowy na M3) |
| 5 | preflight: SQL, model, tracing — tabela „działa / nie działa” |

**Środowisko:** Databricks Free Edition, Serverless. Uruchom notebook przyciskiem **Run all** i czytaj dalej, zanim skończy.

> Dane są syntetyczne i spseudonimizowane: `customer_name` = `Customer <id>`, `tax_id` ma fikcyjne wartości. Pochodzą z datasetu Databricks Marketplace, który prowadzący przygotował wcześniej.

In [ ]:
%pip install --quiet -r ../requirements.txt

In [ ]:
# source: WS3[2]
dbutils.library.restartPython()

In [ ]:
# source: new + WS4[3] + WS2[6]
# Wspólna konfiguracja warsztatu — ta sama komórka jest w każdym notebooku.
CATALOG = "workspace"
SCHEMA = "default"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_customer_360"
VOLUME = "retail_docs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DOCS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_docs"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_chunks"
SEARCH_ENDPOINT = "retail_rag_search"
SEARCH_INDEX = f"{CATALOG}.{SCHEMA}.retail_rag_chunks_index"
AVG_VALUE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_average_customer_value"
PROFILE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_customer_profile"
FORMAT_FUNCTION = f"{CATALOG}.{SCHEMA}.format_customer_for_agent"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"
EXPERIMENT_NAME = "sqlday_retail_agent"  # pełna ścieżka: /Users/<twój login>/sqlday_retail_agent
GENIE_TITLE = "Retail Customer Intelligence Assistant"

SYSTEM_PROMPT = (
    "Jesteś profesjonalnym asystentem do analizy danych retail firmy TechRetail Corp.\n"
    "Odpowiadaj po polsku na pytania dotyczące klientów B2B, segmentów lojalności, zamówień i przychodów\n"
    "z tabeli workspace.default.gold_customer_360 oraz raportów analityków.\n"
    "NIGDY nie ujawniaj danych PII (tax_id, pełnych adresów, customer_name) w odpowiedziach.\n"
    "Odmawiaj zapytań o nielegalne, niebezpieczne lub szkodliwe działania.\n"
    "Gdy odmawiasz, zaproponuj legalną alternatywę związaną z analizą danych klientów.\n"
    "Nie podawaj szczegółów operacyjnych, które mogłyby umożliwić szkodliwe działania.\n"
    "Liczby podawaj wyłącznie z wyników narzędzi; niczego nie zgaduj.\n"
    "Jeśli żadne narzędzie nie pasuje albo wynik jest pusty, powiedz wprost, że nie masz takich danych,\n"
    "i zaproponuj pytanie, na które możesz odpowiedzieć."
)

In [ ]:
# source: new
import os
from pathlib import Path

# Notebook leży w workshop/00_setup/, dane w workshop/data/ tego samego folderu Git.
DATA_DIR = Path(os.getcwd()).parent / "data"
USERNAME = spark.sql("SELECT current_user()").first()[0]

missing = [p for p in ("tables/gold_customer_360.parquet", "checkpoints/retail_rag_chunks.parquet", "documents") if not (DATA_DIR / p).exists()]
assert not missing, f"Brak plików w {DATA_DIR}: {missing}. Zaimportuj repozytorium jako folder Git (README, prework)."
print(f"Użytkownik: {USERNAME}\nDane: {DATA_DIR}")

<!-- source: WS1[15] -->
## 1. Tabela Gold: `gold_customer_360`

Jeden wiersz to jeden klient B2B z cechami RFM: **R**ecency (dni od ostatniego zakupu), **F**requency (liczba pozycji), **M**onetary (wartość zakupów). Do tego segment lojalności 0–3 i lokalizacja. W WS1 tabela powstawała z Marketplace, a tu wczytujesz gotowy wynik z pliku.

Zapis jako tabela Delta w Unity Catalog daje transakcje ACID, wersjonowanie (Time Travel) i uprawnienia, z których skorzystamy w M4.

In [ ]:
# source: new + WS1[16]
import pandas as pd
from pyspark.sql import functions as F

gold_pdf = pd.read_parquet(DATA_DIR / "tables" / "gold_customer_360.parquet")
gold_df = (
    spark.createDataFrame(gold_pdf)
    .withColumn("first_order_date", F.col("first_order_date").cast("date"))
    .withColumn("last_order_date", F.col("last_order_date").cast("date"))
)

(gold_df
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)
print(f"Tabela zapisana: {GOLD_TABLE}")
print(f"  Wiersze: {spark.table(GOLD_TABLE).count():,} (oczekiwane: 28,813)")
print(f"  Kolumny: {len(spark.table(GOLD_TABLE).columns)} (oczekiwane: 19)")

In [ ]:
%sql
-- source: WS1[5]
-- Segmenty lojalności: 0=Nowi/Nieaktywni, 1=Rozwijający się, 2=Regularni, 3=VIP
SELECT
  loyalty_segment,
  COUNT(*)                        AS klienci,
  ROUND(AVG(monetary), 2)         AS avg_monetary,
  ROUND(AVG(recency_days), 0)     AS avg_recency_days,
  SUM(CASE WHEN num_orders = 0 THEN 1 ELSE 0 END) AS bez_zamowien
FROM workspace.default.gold_customer_360
GROUP BY loyalty_segment
ORDER BY loyalty_segment

In [ ]:
%sql
-- source: WS1[6]
-- Geografia: gdzie są klienci?
SELECT state, COUNT(*) AS klienci, ROUND(AVG(monetary), 2) AS avg_monetary
FROM workspace.default.gold_customer_360
GROUP BY state
ORDER BY klienci DESC
LIMIT 10

In [ ]:
# source: WS1[11]
# ai_query(): wywołanie modelu bezpośrednio w SQL — 4 wiersze = 4 wywołania.
display(spark.sql(f"""
WITH segment_stats AS (
  SELECT loyalty_segment, COUNT(*) AS customers, ROUND(AVG(monetary), 2) AS avg_monetary
  FROM {GOLD_TABLE}
  GROUP BY loyalty_segment
)
SELECT
  loyalty_segment,
  customers,
  avg_monetary,
  ai_query(
    '{LLM_ENDPOINT}',
    CONCAT(
      'Jesteś analitykiem retail. Segment lojalności ', CAST(loyalty_segment AS STRING),
      ' ma ', CAST(customers AS STRING), ' klientów, średnia wartość zakupów: ', CAST(avg_monetary AS STRING), ' USD. ',
      'Segment 0=nowy, 1=okazjonalny, 2=regularny, 3=VIP. ',
      'Podaj JEDNĄ krótką rekomendację biznesową (max 20 słów) po polsku.'
    )
  ) AS ai_recommendation
FROM segment_stats
ORDER BY loyalty_segment
"""))

In [ ]:
%sql
-- source: WS1[17]
-- Time Travel: każdy zapis tworzy nową wersję tabeli
DESCRIBE HISTORY workspace.default.gold_customer_360

<!-- source: WS3[3] -->
## 2. Raporty PDF w Volume

Analitycy TechRetail przygotowali 10 raportów o segmentach, geografii, retencji, wartości klientów, churnie i jakości danych. Wrzucamy je do **Volume**, czyli przestrzeni na pliki zarządzanej przez Unity Catalog. W M3 zamienią się w wiedzę, z której korzysta asystent.

In [ ]:
# source: WS3[7]
import shutil

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

# Serverless nie obsługuje dbutils.fs.cp z file: — kopiujemy przez system plików Volume.
pdfs = sorted((DATA_DIR / "documents").glob("*.pdf"))
for pdf in pdfs:
    shutil.copy(pdf, Path(VOLUME_PATH) / pdf.name)

print(f"Pliki w {VOLUME_PATH}:")
for f in sorted(dbutils.fs.ls(VOLUME_PATH), key=lambda x: x.name):
    if f.name.endswith(".pdf"):
        print(f"   • {f.name} ({f.size / 1024:,.0f} KB)")
assert len(pdfs) == 10, f"Oczekiwano 10 PDF, jest {len(pdfs)}"

<!-- source: WS3[13] -->
## 3. Fragmenty raportów: `retail_rag_chunks`

Model nie czyta całych PDF. Wyszukiwarka zwraca mu kilka krótkich **chunków**, czyli fragmentów po ok. 600 znaków. W M3 zobaczysz, jak powstają (`ai_parse_document` → tekst → podział). Tu wczytujesz gotowy wynik, żeby indeks AI Search mógł powstać od razu.

Tabela potrzebuje **klucza głównego** i **Change Data Feed**. Dzięki temu indeks synchronizuje tylko zmienione wiersze.

In [ ]:
# source: WS3[16]
chunks_pdf = pd.read_parquet(DATA_DIR / "checkpoints" / "retail_rag_chunks.parquet")
spark.createDataFrame(chunks_pdf).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(CHUNKS_TABLE)

spark.sql(f"ALTER TABLE {CHUNKS_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
spark.sql(f"ALTER TABLE {CHUNKS_TABLE} ALTER COLUMN chunk_id SET NOT NULL")
try:
    spark.sql(f"ALTER TABLE {CHUNKS_TABLE} ADD CONSTRAINT pk_chunk_id PRIMARY KEY (chunk_id)")
except Exception as e:
    if "already exists" not in str(e).lower():
        raise

n_chunks = spark.table(CHUNKS_TABLE).count()
print(f"{CHUNKS_TABLE}: {n_chunks} chunków z 10 raportów (PK chunk_id, CDF włączony)")
display(spark.table(CHUNKS_TABLE).groupBy("doc_id").count().orderBy("doc_id"))

<!-- source: WS3[8] -->
## 4. Endpoint AI Search (dawniej Vector Search)

**AI Search** to zarządzana wyszukiwarka wektorowa i pełnotekstowa Databricks. Do połowy 2026 roku nazywała się Vector Search. Pierwsze uruchomienie endpointu trwa od kilku do kilkunastu minut, dlatego startujemy go teraz i **nie czekamy**. Do M3 będzie gotowy.

Na Free Edition masz **jeden endpoint**. Jeśli już istnieje, ta komórka go nie rusza.

In [ ]:
# source: new
from databricks.ai_search.client import AISearchClient

search_client = AISearchClient(disable_notice=True)
try:
    if search_client.endpoint_exists(SEARCH_ENDPOINT):
        state = search_client.get_endpoint(SEARCH_ENDPOINT).get("endpoint_status", {}).get("state")
        print(f"Endpoint {SEARCH_ENDPOINT} już istnieje (stan: {state})")
    else:
        search_client.create_endpoint(name=SEARCH_ENDPOINT, endpoint_type="STANDARD")
        print(f"Uruchamiam endpoint {SEARCH_ENDPOINT} w tle — stan pokaże preflight poniżej")
except Exception as e:
    print(f"Nie udało się uruchomić endpointu: {type(e).__name__}: {e}")
    print("Nic straconego: M3 ma tryb offline (retrieve_local) na przygotowanych embeddingach.")

<!-- source: new -->
## 5. Preflight

Sprawdzamy rzeczy, na których opiera się reszta dnia: dane TechRetail, model, tracing, AI Search i drugą domenę (Bakehouse) do zadań poziomu 2 oraz capstone. Jeśli któraś jest na czerwono, zgłoś to prowadzącemu **teraz**, a nie w M3.

In [ ]:
# source: new
import mlflow
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
results = []


def check(name, fn):
    try:
        results.append((name, True, fn()))
    except Exception as e:
        results.append((name, False, f"{type(e).__name__}: {str(e)[:160]}"))


check("SQL na tabeli Gold", lambda: f"{spark.table(GOLD_TABLE).where('loyalty_segment = 3').count():,} klientów VIP (oczekiwane 9,541)")
check("Chunki raportów", lambda: f"{spark.table(CHUNKS_TABLE).count()} wierszy")


def call_llm():
    client = w.serving_endpoints.get_open_ai_client()
    reply = client.chat.completions.create(
        model=LLM_ENDPOINT,
        messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": "Odpowiedz jednym słowem: gotowy?"}],
        max_tokens=10,
    )
    return f"{LLM_ENDPOINT}: {reply.choices[0].message.content.strip()!r}"


check("Model (Foundation Model API)", call_llm)


def trace():
    mlflow.set_experiment(f"/Users/{USERNAME}/{EXPERIMENT_NAME}")
    with mlflow.start_span(name="preflight") as span:
        span.set_inputs({"check": "tracing"})
        span.set_outputs({"ok": True})
    return f"eksperyment /Users/{USERNAME}/{EXPERIMENT_NAME}"


check("MLflow Tracing", trace)


def search_state():
    from databricks.ai_search.client import AISearchClient
    client = AISearchClient(disable_notice=True)
    return client.get_endpoint(SEARCH_ENDPOINT).get("endpoint_status", {}).get("state", "?")


check("Endpoint AI Search (może być PROVISIONING)", search_state)
check("Druga domena: samples.bakehouse (wzorce, capstone)",
      lambda: f"{spark.table('samples.bakehouse.sales_transactions').count():,} transakcji, "
              f"{spark.table('samples.bakehouse.media_customer_reviews').count():,} opinii")

print(f"{'':2} {'Sprawdzenie':<44} Wynik")
for name, ok, detail in results:
    print(f"{'✅' if ok else '❌'} {name:<44} {detail}")

<!-- source: new -->
## Lab: poznaj dane, zanim zapytasz o nie model

**Na koniec:** otwórz `workshop/transfer/canvas_agenta.md` i wpisz domenę, dla której chcesz zbudować agenta pod koniec dnia, oraz 5 pytań jej użytkowników. Jeśli nie masz własnej domeny, wpisz „sieć piekarni Bakehouse”.

Odpowiedz na trzy pytania **samym SQL**. Dodaj komórkę poniżej i zapisz odpowiedzi. Te same pytania w M1 zadasz modelowi bez dostępu do danych, a w M5 agentowi z narzędziami.

1. Ilu jest klientów VIP (`loyalty_segment = 3`) i jaka jest ich średnia `monetary`?
2. Który stan ma najwięcej klientów?
3. Jaki odsetek klientów nie złożył żadnego zamówienia (`num_orders = 0`)?

<details><summary>Oczekiwane wyniki (sprawdź po swojej próbie)</summary>

1. 9 541 klientów VIP, średnia `monetary` ≈ 1038,72 USD
2. NY, 3 417 klientów
3. 26 862 z 28 813, czyli ok. 93%. Większość bazy to klienci bez historii zamówień w oknie danych.

</details>

<!-- source: new -->
## Podsumowanie

- `gold_customer_360` to **dane ustrukturyzowane**: liczby, na które odpowiada SQL.
- `retail_docs` i `retail_rag_chunks` to **wiedza nieustrukturyzowana**: wnioski i rekomendacje, których w tabeli nie ma.
- Asystent, którego budujemy, musi umieć skorzystać z obu źródeł i wiedzieć, kiedy żadne nie pasuje.

**Dalej:** `demo/m1_agentic_ai_playground` (prowadzący) i `labs/m1_agentic_ai_playground` (Ty).